# Module 1 · Lesson 07: Response Metadata Inspector

Every API call returns **much more than just text**. The response object contains
rich metadata that is essential for production systems.

## What you will learn
1. The **anatomy** of a response object
2. Token usage and **cost estimation**
3. **finish_reason** — detecting truncated outputs
4. **Log-probabilities** — measuring model confidence
5. **Reproducibility** — temperature, seed, system fingerprint
6. **Latency profiling** — benchmarking API speed

In [1]:
# ── Setup ──────────────────────────────────────────────
import os, time, math, json
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI
client = OpenAI()

# Pricing per 1M tokens (early 2025)
PRICING = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

def estimate_cost(model, in_tok, out_tok):
    p = PRICING.get(model, PRICING["gpt-4o-mini"])
    return (in_tok / 1e6) * p["input"] + (out_tok / 1e6) * p["output"]

print("✅ Ready")

✅ Ready


---
## 1. Response Object Anatomy

Let's make a call and inspect every field in the response:

In [6]:
# ── Section 1: Response anatomy ───────────────────────
prompt = "Write a haiku about debugging."
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=50
)

# Pretty-print the structure
created_dt = datetime.fromtimestamp(response.created, tz=timezone.utc)

anatomy = f"""
### Response Object Structure

| Field | Value |
|-------|-------|
| **id** | `{response.id}` |
| **model** | `{response.model}` |
| **object** | `{response.object}` |
| **created** | {response.created} → {created_dt.strftime('%Y-%m-%d %H:%M:%S UTC')} |
| **system_fingerprint** | `{response.system_fingerprint}` |
| **finish_reason** | `{response.choices[0].finish_reason}` |
| **prompt_tokens** | {response.usage.prompt_tokens} |
| **completion_tokens** | {response.usage.completion_tokens} |
| **total_tokens** | {response.usage.total_tokens} |

**Response text:** {response.choices[0].message.content}
"""
display(Markdown(anatomy))

print("💡 The 'id' uniquely identifies this request — useful for logging and debugging.")
print("   The 'system_fingerprint' tells you which backend served your request.")


### Response Object Structure

| Field | Value |
|-------|-------|
| **id** | `chatcmpl-DHnQPrXnzo9L8Icfx8Hu7mSj5XZ4Y` |
| **model** | `gpt-4o-mini-2024-07-18` |
| **object** | `chat.completion` |
| **created** | 1773133937 → 2026-03-10 09:12:17 UTC |
| **system_fingerprint** | `fp_a1ddba3226` |
| **finish_reason** | `stop` |
| **prompt_tokens** | 14 |
| **completion_tokens** | 17 |
| **total_tokens** | 31 |

**Response text:** Code whispers secrets,  
Errors hide in shadows deep,  
Logic's light reveals.


💡 The 'id' uniquely identifies this request — useful for logging and debugging.
   The 'system_fingerprint' tells you which backend served your request.


---
## 2. Token Usage & Cost Deep Dive

Input tokens are always cheaper than output tokens. Let's see how prompt length affects cost:

In [ ]:
# ── Section 2: Token usage comparison ─────────────────
prompts = [
    ("Short prompt",  "What is Python?"),
    ("Medium prompt", "Explain what Python is, its main features, and why it is popular for AI."),
    ("Long prompt",   "You are an expert software engineer. Based on your extensive experience "
                      "with multiple programming languages including Java, C++, Ruby, and Go, "
                      "provide a detailed comparison of Python's strengths and weaknesses "
                      "for building web applications, data science pipelines, and CLI tools."),
]

print(f"{'Prompt':<16} {'In Tok':<8} {'Out Tok':<8} {'Total':<8} {'Cost':>10} {'Cost/1K':>12}")
print("─" * 64)

for label, prompt in prompts:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )
    cost = estimate_cost("gpt-4o-mini", r.usage.prompt_tokens, r.usage.completion_tokens)
    print(f"{label:<16} {r.usage.prompt_tokens:<8} {r.usage.completion_tokens:<8} "
          f"{r.usage.total_tokens:<8} ${cost:>9.6f} ${cost*1000:>11.4f}")

print("\n💡 Longer prompts = more input tokens = higher cost. Optimize prompt length!")

---
## 3. finish_reason — Detecting Truncation

| finish_reason | Meaning | Action |
|---------------|---------|--------|
| `stop` | Model finished naturally | ✅ All good |
| `length` | Hit `max_tokens` limit | ⚠️ Response was truncated! |
| `content_filter` | Blocked by safety filters | 🛡️ Review the prompt |

In [11]:
# ── Section 3: Finish reason ──────────────────────────
# Natural stop
r1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'hello' and nothing else."}],
    max_tokens=100
)

# Forced truncation
r2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a 500-word essay about space exploration."}],
    max_tokens=15   # Intentionally very low!
)

# print(f"{'Scenario':<28} {'finish_reason':<16} {'Response Preview'}")
# print("-" * 70)
# print(f"{'Natural completion':<28} {r1.choices[0].finish_reason:<16} {r1.choices[0].message.content[:50]}")
# print(f"{'Truncated (max_tokens=15)':<28} {r2.choices[0].finish_reason:<16} {r2.choices[0].message.content[:50]}...")

print("Natural completion:")
print("finish_reason:", r1.choices[0].finish_reason)
print("response:", r1.choices[0].message.content[:50])

print("\nTruncated (max_tokens=15):")
print("finish_reason:", r2.choices[0].finish_reason)
print("response:", r2.choices[0].message.content[:50], "...")

print("\n⚠️ Always check finish_reason in production to detect truncated outputs!")

Natural completion:
finish_reason: stop
response: Hello!

Truncated (max_tokens=15):
finish_reason: length
response: Space exploration has long captured the imaginatio ...

⚠️ Always check finish_reason in production to detect truncated outputs!


---
## 4. Log-Probabilities — Token Confidence

**Logprobs** show how *confident* the model is about each token choice.
A probability of 99% means the model is very sure; 30% means it considered alternatives.

In [14]:
# ── Section 4: Logprobs ───────────────────────────────
r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Be concise."},
        {"role": "user", "content": "What is the capital of Japan?"}
    ],
    max_tokens=30,
    logprobs=True,
    top_logprobs=3   # Show top 3 alternatives per token
)

print(f"Response: {r.choices[0].message.content}\n")

if r.choices[0].logprobs and r.choices[0].logprobs.content:
    print(f"{'#':<4} {'Token':<15} {'LogProb':<10} {'Prob':<10} {'Alternatives'}")
    print("─" * 70)

    for i, tok_info in enumerate(r.choices[0].logprobs.content):
        prob = math.exp(tok_info.logprob) * 100

        # Format alternatives
        alts = ""
        if tok_info.top_logprobs:
            alt_parts = [f"'{a.token}' ({math.exp(a.logprob)*100:.1f}%)" for a in tok_info.top_logprobs[:3]]
            alts = " | ".join(alt_parts)

        conf = "🟢" if prob > 90 else ("🟡" if prob > 50 else "🔴")
        print(f"{i+1:<4} {repr(tok_info.token):<15} {tok_info.logprob:<10.3f} {conf}{prob:>5.1f}%   {alts}")

print("\n💡 Use logprobs for: classification confidence, hallucination detection, output quality scoring.")

Response: The capital of Japan is Tokyo.

#    Token           LogProb    Prob       Alternatives
──────────────────────────────────────────────────────────────────────
1    'The'           -0.014     🟢 98.6%   'The' (98.6%) | 'Tokyo' (1.4%) | 'Tok' (0.0%)
2    ' capital'      0.000      🟢100.0%   ' capital' (100.0%) | 'capital' (0.0%) | ' Capital' (0.0%)
3    ' of'           0.000      🟢100.0%   ' of' (100.0%) | 'of' (0.0%) | ' city' (0.0%)
4    ' Japan'        0.000      🟢100.0%   ' Japan' (100.0%) | 'Japan' (0.0%) | ' japan' (0.0%)
5    ' is'           0.000      🟢100.0%   ' is' (100.0%) | ' Is' (0.0%) | ' هو' (0.0%)
6    ' Tokyo'        -0.001     🟢 99.9%   ' Tokyo' (99.9%) | 'Tokyo' (0.1%) | ' Tokio' (0.0%)
7    '.'             -0.000     🟢100.0%   '.' (100.0%) | '.
' (0.0%) | '.

' (0.0%)

💡 Use logprobs for: classification confidence, hallucination detection, output quality scoring.


---
## 5. Reproducibility — Getting the Same Answer Twice

Use `temperature=0` + `seed` for deterministic outputs. But same `system_fingerprint` is also needed.

In [ ]:
# ── Section 5: Reproducibility ────────────────────────
prompt = "What is 2 + 2? Answer with just the number."

runs = []
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.0,
        seed=42
    )
    runs.append({
        "content": r.choices[0].message.content.strip(),
        "fingerprint": r.system_fingerprint,
    })

print(f"{'Run':<6} {'Response':<12} {'System Fingerprint'}")
print("─" * 50)
for i, run in enumerate(runs, 1):
    print(f"{i:<6} {run['content']:<12} {run['fingerprint']}")

all_same = len(set(r['content'] for r in runs)) == 1
same_fp  = len(set(r['fingerprint'] for r in runs)) == 1

print(f"\n{'✅ All responses identical!' if all_same else '⚠️ Responses varied!'}")
print(f"{'   Same backend (fingerprint match)' if same_fp else '   Different backends — may cause variations'}")

---
## 6. Latency Profiling

In [ ]:
# ── Section 6: Latency profiling ──────────────────────
tests = [
    ("Short (10 tokens)",   10,  "Say hello."),
    ("Medium (100 tokens)", 100, "Explain what Python is."),
    ("Long (300 tokens)",   300, "Write a short tutorial about lists in Python."),
]

print(f"{'Test':<24} {'Latency (ms)':<14} {'Out Tokens':<12} {'Tokens/sec'}")
print("─" * 60)

for label, max_tok, prompt in tests:
    start = time.perf_counter()
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok
    )
    ms = (time.perf_counter() - start) * 1000
    out = r.usage.completion_tokens
    tps = out / (ms / 1000) if ms > 0 else 0
    print(f"{label:<24} {ms:<14.0f} {out:<12} {tps:.1f}")

print("\n💡 More output tokens → more latency. Streaming hides this!")

---
## Key Takeaways 📝

| Concept | Why It Matters |
|---------|----------------|
| **Response ID** | Unique per request — essential for logging |
| **finish_reason** | Detect truncated or filtered responses |
| **Token usage** | Monitor costs in production |
| **Logprobs** | Measure model confidence per token |
| **System fingerprint** | Track backend changes affecting reproducibility |
| **temperature + seed** | Enable deterministic outputs |
| **Latency ∝ output length** | Longer outputs = slower responses |

---
**🎉 Module 1 Complete!** You now understand the fundamentals of LLM APIs.

**Next Module:** `module_02_prompt_engineering` — Master the art of writing effective prompts